# AST 326: Asteroid Orbit Lab 
### Tutorial 2 - Deriving an orbit from observations
- Primary Authors: Bethany Ludwig, Jacob Taylor
- Last Revision: March 2024
  
### Overview of the Lab:  

The goal of this lab is to calculate the orbit of the asteroid 26 Proserpina. 

This lab is broken into three tutorials: 

<span style="color:red"> 1) Understand the physics of orbital mechanics: </span>
- Work though the details of orbital motion and understand the definitions of the 
Keplerian orbital elements and ecliptic and equatorial coordinates.
- Take the orbital elements for Ceres in Table 4 and compute and plot the orbital separation, r, the
true anomaly, v, and the x- and y-coordinates in the orbital plane as a function of time. 
- Reproduce Figure 3, Figure 4, and Figure 5 in the lab manual. 

<span style="color:red"> 2) Apply what you learned with Ceres to 26 Proserpina:   </span>
- Get the position of the asteroid at the times of observation for Lab #4.   
- For the first three epochs, convert the measured coordinates from equatorial to Cartesian   
ecliptic coordinates and find the components of the target unit vector s.  
- Compute the first and second time derivatives of s using a Taylor series approximation.  
- Compute the heliocentric Cartesian ecliptic coordinates of the earth on your observation dates.  
- Compute r, ρ, and dρ/dt for your asteroid.  
- Compute r and dr/dt.  
- Derive the Keplerian orbital elements from r and dr/dt  

<span style="color:red"> 3) Analyze the orbit you determined:   </span>
- Use your estimated orbital elements to predict the geocentric equatorial coordinates of the 
asteroid at the fourth epoch and compare the measured and predicted positions of the 
asteroid. Discuss the errors in the orbital elements and your predicted position.  
-  If time permits, figure out how to combine data from more than three epochs to estimate
the orbital elements.  
  

### Step 1: Get the position of the asteroid at the times of observation for Lab #4.

In [1]:
# Enter the observing times for your asteroid.
jd =  [ ... ]

# Get the Ra/Dec coordinates for your asteroid at the observing times.
# You can use the ones you calculated in Lab 4, you can use the Horizons database, or some other method.
# Be clear in your lab report about what you used and how you accessed/measured it. 
from astroquery.jplhorizons import Horizons
obj = Horizons(id='A853 JA',  epochs=jd, location='500', id_type='smallbody')
table = obj.ephemerides()
table['targetname','datetime_str','datetime_jd','RA','DEC']

targetname,datetime_str,datetime_jd,RA,DEC
---,---,d,deg,deg
str23,str24,float64,float64,float64
26 Proserpina (A853 JA),2012-Jan-20 07:27:54.000,2455946.811041667,81.02926,26.98072
26 Proserpina (A853 JA),2012-Jan-21 07:39:37.000,2455947.81917824,80.88885,26.9684
26 Proserpina (A853 JA),2012-Jan-23 08:35:52.000,2455949.85824074,80.62674,26.9431
26 Proserpina (A853 JA),2012-Jan-26 05:05:05.000,2455952.71186343,80.31025,26.90734


### Step 2: Convert the measured coordinates from equatorial to Cartesian ecliptic coordinates and find the components of the target unit vector s. 

In [2]:
# Convert from celestial coordinates to equatorial coordinates Eq 61 
import numpy as np 

# Aka alpha, delta
ra = np.deg2rad(table['RA'])
dec = np.deg2rad(table['DEC'])

x = 
y = 
z = 
S_equatorial = 

# Convert from equatorial coordinates to ecliptic Eq 62 - 63  
epsilon = np.deg2rad(23.43929111) # The angle between the equatorial plane and the ecliptic plane for equinox J2000.
Tx = np.array([[ ... ], [ ... ], [ ...]])
S_ecliptic = np.dot(Tx, S_equatorial)

### Step 3: Compute the first and second time derivatives of s using a Taylor series approximation.  

In [3]:
# Eq 50: Get the first time derivatives of the position vector 
# Unless you're looping through each coordinate individually, the output of S_ecliptic might not give you what you expect
# Print out the table and see if it makes sense. If not, you might need to transpose it with ".T"
s1 = S_ecliptic.T[0]
s2 = S_ecliptic.T[1]
s3 = S_ecliptic.T[2]
t = table['datetime_jd']
tau1 = 
tau3 = 
ds_dt = 

# Eq 50: Get the second time derivative of the position vector 
d2s_dt2 =

### Step 4: Compute the heliocentric Cartesian ecliptic coordinates of the earth on your observation dates.

In [4]:
# Determines the location of the sun at a given time (or times, if the input is an array Time object), in geocentric coordinates.
from astropy.coordinates import get_sun
from astropy.time import Time

# To use the get_sun function, we need to convert the Julian date to a Time object
obs_time = Time(table['datetime_jd'],format='jd')

# Get the solar coordinates at the observing times
solar_coordinates = get_sun(obs_time)
solar_ra = solar_coordinates.ra.deg
solar_dec = solar_coordinates.dec.deg

# Eq 61: Convert from celestial coordinates to equatorial coordinates 
ra_sun = np.deg2rad(solar_ra)
dec_sun = np.deg2rad(solar_dec)

x_sun = 
y_sun = 
z_sun = 
S_equatorial_sun = np.array([ ... ])

# Convert from equatorial coordinates to ecliptic Eq 62 - 63 
# The Sun - Earth vector  should be -X, +Y, -Z
S_ecliptic_sun =

### Step 5: Compute r, ρ, and dρ/dt for your asteroid.

In [5]:
# We now want the orbital velocity with respect to the sun
import astropy.constants as const 
import astropy.units as u

# Define k (page 2)
k = np.sqrt(const.G * const.M_sun).to(u.AU**(3/2)/u.day).value

# Eq 47: We need to iteratively solve to get rho, which means we need a function
# You need a set of 3 observations to get the second derivatives, 
# So we'll center the calculation in the middle 
def calculate_rho(r,R=, s=, ds_dt=, d2s_dt2=, k=):
    # Use np.linalg.norm to get the magnitude of a vector
    R_magnitude = np.linalg.norm(S_ecliptic_sun)
    rho = k**2 * (1/R_magnitude**(3) - 1/r**(3)) * (np.dot(ds_dt,np.cross(R,s)) / np.dot(ds_dt,np.cross(d2s_dt2,s)))
    return rho

# I'm choosing to use newton's method to solve for rho 
# Whatever way you solve this be sure to discuss it in your lap report.
from scipy.optimize import fsolve

r = 1 # Initial guess for r
rho = fsolve( ... )

# Eq 48: Now that we have rho we can calculate r 
R_magnitude = np.linalg.norm(S_ecliptic_sun)
R = 
r = 

# Eq 49: dp/dt 
drho_dt = 

### Step 6: Compute r and dr/dt.  

In [6]:
# Eq 41: The heliocentric position of the asteroid
r_vec = 

# Eq 50: et dR/dt the same way we did before
sun1 = 
sun2 = 
sun3 = 
dsun_earth_dt = 

# Eq 46: The heliocentric velocity of the asteroid
dr_dt =  

### Step 7: Derive the Keplerian orbital elements from r and dr/dt  

In [ ]:
# Eq 51: At the midpoint of our three asteroid observations we know the magnitude of the radius vector, r, and the total velocity, V. 
r_mag = 
V_mag = 

# Semimajor axis
a = 

# Eq 5: Compute h the specific angular momentum
h = 

# Eq 54: Get Omega 
omega = 

# Eq 55: Get i
h_mag = np.linalg.norm(h)
inclination = 

# Eq 56: Get e
eccentricity = 

# Eq 57: Get eccentric anomaly, the first few lines here help with the sign issue  
theta = np.arccos(r_vec[2]/r_mag)
phi = np.arctan2(r_vec[1],r_vec[0])
e_r = np.array([np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta)])
v_r = np.dot(dr_dt, e_r)
ecc_anomaly = np.arccos((a-r_mag)/(a*eccentricity))*np.sign(v_r)

# Eq 25: Get the mean anomaly
mean_anomaly = 

# Get Mean Motion
n = 

# Eq 26: Get tau 
tau = 

# Orbital period
p = 

# Eq 31: True Anomaly 
true_anomaly = 

# Eq 58: Get Argument of Perihelion
arg_perihelion = 